In [0]:
%pip install geopy

In [0]:
pip install geopandas

In [0]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import pandas as pd
from pyspark.sql import functions as F

SILVER_STATIC_PATH = (
    "abfss://wildfire-data@stscewildfirevr01.dfs.core.windows.net/"
    "silver/zone_static_features"
)

LOCATION_LOOKUP_PATH = (
    "abfss://wildfire-data@stscewildfirevr01.dfs.core.windows.net/"
    "silver/zone_location_lookup"
)

In [0]:
zones = (
    spark.read
    .format("delta")
    .load(SILVER_STATIC_PATH)
    .select(
        "zone_id",
        "geometry_wkt"
    )
    .toPandas()
)

print("Zones:", len(zones))

In [0]:
import geopandas as gpd
from shapely import wkt

# Convert stored WKT back into geometry
zones["geometry"] = zones["geometry_wkt"].apply(wkt.loads)

zones_gdf = gpd.GeoDataFrame(
    zones[["zone_id", "geometry"]],
    geometry="geometry",
    crs="EPSG:3310"
)

# Pick a point guaranteed to fall inside each zone
zone_points = zones_gdf.copy()
zone_points["geometry"] = zone_points.geometry.representative_point()

# Convert to latitude / longitude
zone_points = zone_points.to_crs("EPSG:4326")

zone_points["longitude"] = zone_points.geometry.x
zone_points["latitude"] = zone_points.geometry.y

location_lookup = zone_points[
    ["zone_id", "latitude", "longitude"]
].copy()

print("Location rows:", len(location_lookup))
print("Unique zones:", location_lookup["zone_id"].nunique())

display(location_lookup.head(10))

In [0]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

geolocator = Nominatim(
    user_agent="western_la_wildfire_project"
)

reverse = RateLimiter(
    geolocator.reverse,
    min_delay_seconds=1.1,
    swallow_exceptions=True
)

def get_location_name(row):
    location = reverse(
        (row["latitude"], row["longitude"]),
        language="en",
        zoom=14,
        addressdetails=True
    )

    if location is None:
        return "Los Angeles County"

    address = location.raw.get("address", {})

    for key in [
        "city",
        "town",
        "village",
        "suburb",
        "neighbourhood",
        "hamlet",
        "locality"
    ]:
        if address.get(key):
            return address[key]

    return "Los Angeles County"


location_lookup["location_name"] = location_lookup.apply(
    get_location_name,
    axis=1
)

print(
    location_lookup["location_name"]
    .value_counts()
    .head(20)
)

print(
    "Missing location names:",
    location_lookup["location_name"].isna().sum()
)

In [0]:
location_lookup["zone_label"] = (
    location_lookup["location_name"]
    + " — "
    + location_lookup["zone_id"]
)

location_lookup["google_maps_url"] = (
    "https://www.google.com/maps/search/?api=1&query="
    + location_lookup["latitude"].astype(str)
    + ","
    + location_lookup["longitude"].astype(str)
)

display(
    location_lookup[
        [
            "zone_id",
            "location_name",
            "zone_label",
            "latitude",
            "longitude",
            "google_maps_url"
        ]
    ].head(10)
)

In [0]:
location_lookup_spark = spark.createDataFrame(
    location_lookup[
        [
            "zone_id",
            "location_name",
            "zone_label",
            "latitude",
            "longitude",
            "google_maps_url"
        ]
    ]
)

(
    location_lookup_spark.write
    .format("delta")
    .mode("overwrite")
    .save(LOCATION_LOOKUP_PATH)
)

print("Location lookup saved.")
print("Rows:", location_lookup_spark.count())